In [2]:
# Load libraries
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

In [5]:
df = pd.read_csv("../../../data/data_all.csv")

In [7]:
df.head()

,Unnamed: 0,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


R code was essentially TEMP + PRES + RAIN + WSPM * wd_coarse. `model.matrix()` automatically expands those categorical variables and interactions but in Python I have to do it myself.

## Build the Predictor Matrix X 

In [8]:
mapping = {
    "ENE": "E",
    "ESE": "E",
    "NNE": "NE",
    "NNW": "NW",
    "SSE": "SE",
    "SSW": "SW",
    "WNW": "W",
    "WSW": "W"
}

df["wd_coarse"] = df["wd"].replace(mapping)

In [9]:
df["wd_coarse"] = df["wd_coarse"].astype("category")

In [10]:
df["wd_coarse"].value_counts()

wd_coarse
E     80623
NE    64915
W     56285
NW    53194
SW    47052
SE    34422
N     27994
S     17683
Name: count, dtype: int64

In [11]:
# numeric predictors
X_num = df[["TEMP", "PRES", "RAIN", "WSPM"]]

# categorical variable
wd_dummies = pd.get_dummies(df["wd_coarse"], drop_first=False)

### Create Interaction term WSPM * winddirection

In [12]:
WSPM = df["WSPM"].values.reshape(-1,1)
interaction = WSPM * wd_dummies.values
interaction = pd.DataFrame(interaction,
                           columns=[f"WSPM_{c}" for c in wd_dummies.columns])

### Combine Everything

In [13]:
X_df = pd.concat([X_num, wd_dummies, interaction], axis=1)

X = X_df.values

In [16]:
X = X_df.to_numpy(dtype=np.float32)

## Create Y

In [17]:
Y = np.log(df[["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]].values)

In [18]:
Y

array([[1.38629436, 1.38629436, 1.38629436, 1.94591015, 5.70378247,
        4.34380542],
       [2.07944154, 2.07944154, 1.38629436, 1.94591015, 5.70378247,
        4.34380542],
       [1.94591015, 1.94591015, 1.60943791, 2.30258509, 5.70378247,
        4.29045944],
       ...,
       [2.63905733, 3.33220451, 1.38629436, 3.63758616, 6.2146081 ,
        3.98898405],
       [2.48490665, 3.13549422, 1.38629436, 3.40119738, 5.99146455,
        4.07753744],
       [2.56494936, 2.94443898, 1.38629436, 3.63758616, 6.39692966,
        3.8918203 ]])

## Index Split

In [19]:
# Index Split
n = X.shape[0]

train_idx = np.arange(0, n, 3)
valid_idx = np.arange(1, n, 3)
test_idx  = np.arange(2, n, 3)

In [20]:
# Apply the index split
X_train = X[train_idx]
X_valid = X[valid_idx]
X_test  = X[test_idx]

Y_train = Y[train_idx]
Y_valid = Y[valid_idx]
Y_test  = Y[test_idx]

## Convert to Tensors for PyTorch

In [21]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
Y_valid_tensor = torch.tensor(Y_valid, dtype=torch.float32)
Y_test_tensor  = torch.tensor(Y_test, dtype=torch.float32)

## Neural Network

### Define in a Variable

In [22]:
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 6)
)

### Loss Function

In [23]:
loss_fn = nn.MSELoss()

### Optimizer

In [24]:
optimizer = optim.Adam(model.parameters(), lr=0.0005)

### Training loop

In [25]:
print(X_train_tensor.shape)
print(Y_train_tensor.shape)

torch.Size([127390, 20])
torch.Size([127390, 6])


In [26]:
for epoch in range(200):

    preds = model(X_train_tensor)

    loss = loss_fn(preds, Y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [27]:
print(preds.shape)

torch.Size([127390, 6])


### Evaluate Test MSE

In [28]:
test_preds = model(X_test_tensor).detach().numpy()

mse = mean_squared_error(Y_test, test_preds)

print("Test MSE:", mse)

Test MSE: 1.1604409691242494
